In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/anikmajumdar01/cic-darknet2020/Darknet.CSV


# **Clone GitHub repository and install requirements**

In [2]:
!git clone https://github.com/linwhitehat/ET-BERT.git
%cd ET-BERT
!pip install -r requirements.txt

Cloning into 'ET-BERT'...
remote: Enumerating objects: 791, done.
remote: Counting objects: 100% (383/383), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 791 (delta 304), reused 300 (delta 300), pack-reused 408 (from 1)
Receiving objects: 100% (791/791), 13.38 MiB | 26.20 MiB/s, done.
Resolving deltas: 100% (414/414), done.
/kaggle/working/ET-BERT


# **Install additional required libraries**

In [3]:
!pip install torch torchvision torchaudio transformers pandas numpy scikit-learn gdown

# **Verify Environment and GPU Availability**

In [4]:
import torch
from transformers import BertTokenizer, BertModel
print("Current Working Directory:", os.getcwd())
print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

Current Working Directory: /kaggle/working/ET-BERT
PyTorch Version: 2.11.0+cu128
GPU Available: True


In [5]:
!dir

bert_base_config.json  datasets     inference  preprocess.py  requirements.txt
corpora		       fine-tuning  LICENSE    pre-training   uer
data_process	       images	    models     README.md      vocab_process


In [6]:
!cat README.md

# ET-BERT: A Contextualized Datagram Representation with Pre-training Transformers for Encrypted Traffic Classification

<!-- 
[![codebeat badge](https://codebeat.co/badges/f75fab90-6d00-44b4-bb42-d19067400243)](https://codebeat.co/projects/github-com-linwhitehat-et-bert-main) 
![](https://img.shields.io/badge/license-MIT-000000.svg) 
[![arXiv](https://img.shields.io/badge/arXiv-1909.05658-<color>.svg)](https://arxiv.org/abs/2202.06335) 
-->

<p align="center">
  <a href=''><img src='https://img.shields.io/badge/License-MIT-brightgreen'></a> 
  <a href='https://arxiv.org/abs/2202.06335'><img src='https://img.shields.io/badge/arXiv-1909.05658-<color>.svg'></a> 
  <a href='https://dl.acm.org/doi/10.1145/3485447.3512217' target='_blank'><img src="https://img.shields.io/badge/WWW'22-Paper-blue"></a>
</p>

**Note:**
- ⭐ **Please leave a <font color='orange'>STAR</font> if you like this project!** ⭐
- If you find any <font color='red'>incorrect</font> / <font color='red'>inappropriate</font>

# **Download the pre-trained model**

In [7]:
import gdown
import os

os.makedirs('models', exist_ok=True)

file_id = '1r1yE34dU2W8zSqx1FkB8gCWri4DQWVtE'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'models/pretrained_model.bin'

gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1r1yE34dU2W8zSqx1FkB8gCWri4DQWVtE
From (redirected): https://drive.google.com/uc?id=1r1yE34dU2W8zSqx1FkB8gCWri4DQWVtE&confirm=t&uuid=68788efa-c557-4ffc-b95a-f6facd137bcd
To: /kaggle/working/ET-BERT/models/pretrained_model.bin
100%|██████████| 716M/716M [00:02<00:00, 304MB/s]


'models/pretrained_model.bin'

# **Set up the Python path (PYTHONPATH)**

In [8]:
import os
os.environ['PYTHONPATH'] = os.getcwd()
print("PYTHONPATH set to:", os.environ['PYTHONPATH'])

PYTHONPATH set to: /kaggle/working/ET-BERT


# **Load and clean the Darknet.CSV dataset**

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import RandomOverSampler
import os

csv_path = '/kaggle/input/datasets/anikmajumdar01/cic-darknet2020/Darknet.CSV'
df = pd.read_csv(csv_path, low_memory=False)

# Remove infinite and missing values
df = df.replace([np.inf, -np.inf], np.nan).dropna()

# **Encode target labels to integer format**

In [10]:
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['Label'].astype(str))
num_labels = len(le.classes_)
print(f"Total unique labels found: {num_labels}")

Total unique labels found: 11


# **Separate features and apply standard scaling**

In [11]:
# Drop target and ID columns
columns_to_drop = ['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Timestamp', 'Traffic Type', 'Label', 'label_id']
X = df.drop(columns=[col for col in columns_to_drop if col in df.columns])
y = df['label_id']

# Scale the numerical data to a standard range
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# **Balance the dataset to prevent class imbalance**

In [12]:
# This will duplicate minority class samples so the model learns them properly
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_scaled, y)

# **Convert numerical features into text sequences**

In [13]:
# Convert back to dataframe and round to 4 decimal places to limit token size
X_res_df = pd.DataFrame(X_resampled, columns=X.columns).round(4)

# Join row values with a space to create the text sequence
texts = X_res_df.astype(str).agg(' '.join, axis=1).tolist()
etbert_df = pd.DataFrame({'label': y_resampled, 'text_a': texts})

# **Split the dataset and save as TSV files**

In [14]:
train_df, temp_df = train_test_split(etbert_df, test_size=0.2, random_state=42, stratify=etbert_df['label'])
valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

output_dir = 'darknet_tsv'
os.makedirs(output_dir, exist_ok=True)

train_df.to_csv(f'{output_dir}/train_dataset.tsv', sep='\t', index=False)
valid_df.to_csv(f'{output_dir}/valid_dataset.tsv', sep='\t', index=False)
test_df.to_csv(f'{output_dir}/test_dataset.tsv', sep='\t', index=False)

In [15]:
print(f"Train dataset size: {len(train_df)}")

Train dataset size: 426976


In [16]:
print(f"Valid dataset size: {len(valid_df)}")

Valid dataset size: 53372


In [17]:
print(f"Test dataset size: {len(test_df)}")

Test dataset size: 53372


In [18]:
!sed -i 's|/data2/lxj/pre-train/results/confusion_matrix|./confusion_matrix.txt|g' fine-tuning/run_classifier.py

# **Start model fine-tuning with optimized hyperparameters**

In [19]:
import gc
import torch

# Clear GPU memory
gc.collect()
torch.cuda.empty_cache()

!python fine-tuning/run_classifier.py \
    --pretrained_model_path models/pretrained_model.bin \
    --vocab_path models/encryptd_vocab.txt \
    --train_path darknet_tsv/train_dataset.tsv \
    --dev_path darknet_tsv/valid_dataset.tsv \
    --test_path darknet_tsv/test_dataset.tsv \
    --epochs_num 5 \
    --batch_size 32 \
    --embedding word_pos_seg \
    --encoder transformer \
    --mask fully_visible \
    --seq_length 128 \
    --learning_rate 2e-5

Batch size:  32
The number of training instances: 426976
2 GPUs are available. Let's use them.
Start training.
  0%|                                                     | 0/5 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
Epoch id: 1, Training steps: 100, Avg loss: 2.407
Epoch id: 1, Training steps: 200, Avg loss: 2.362
Epoch id: 1, Training steps: 300, Avg loss: 2.234
Epoch id: 1, Training steps: 400, Avg loss: 2.076
Epoch id: 1, Training steps: 500, Avg loss: 1.968
Epoch id: 1, Training steps: 600, Avg loss: 1.853
Epoch id: 1, Training steps: 700, Avg loss: 1.730
Epoch id: 1, Training steps: 800, Avg loss: 1.615
Epoch id: 1, Training steps: 900, Avg loss: 1.567
Epoch id: 1, Training steps: 1000, Avg loss: 1.464
Epoch id: 1, Training steps: 1100, Avg los